# Recurrent 1.5-Bit Concept Bottleneck Model: Master Execution Notebook

This notebook runs the complete pipeline in sequence:
1. **Stage 1**: Supervised Fine-Tuning (PEFT/LoRA SFT) on Qwen 1.5B (distilled DeepSeek-R1) via Unsloth.
2. **Stage 2**: Batched Activation Hooking & Chunked Caching on Layer 14.
3. **Stage 3**: HybridCBM Representation Decomposition & Cosine Similarity Concept Translation.
4. **Stage 4**: T-TRM Recurrent Loop and CMR logic decider joint training under PST (Polynomial Surrogate Training) and monotonicity regularizations.

## 1. Setup Environment & Repository

Clone the code repository (if running in a fresh workspace) and navigate to the project root directory. This ensures all path references are correct.

In [ ]:
import os

# Check if we are already in the repository root
if not os.getcwd().endswith("recurrent-1.5bit-cbm"):
    if not os.path.exists("recurrent-1.5bit-cbm"):
        print("Cloning repository...")
        !git clone https://github.com/Borisz42/recurrent-1.5bit-cbm.git
    
    # Navigate into the repository directory
    %cd recurrent-1.5bit-cbm
    print("Pulling latest repository updates...")
    !git pull
else:
    print("Already in repository root. Pulling latest updates...")
    !git pull

## 2. Install Dependencies

Install the optimized libraries. On Kaggle, Unsloth must be installed using their specific wheels or git repository.

In [ ]:
# Install optimized deep learning and training libraries
!pip install -q lightning pytorch-lightning transformers accelerate safetensors datasets trl bitsandbytes>=0.46.1
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 3. Stage 1: Base Model LoRA SFT

Fine-tune `DeepSeek-R1-Distill-Qwen-1.5B` in 4-bit precision on the instruction dataset. Formats instruction templates with reasoning cognitive traces using `<think>...</think>` tags.

In [ ]:
# Run SFT on the full dataset (runs for 1 epoch)
# Increased batch_size to 8 to utilize more GPU memory and speed up training
!python src/system1/train_sft.py \
    --model_name "unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit" \
    --dataset "tatsu-lab/alpaca" \
    --output_dir "./outputs" \
    --adapter_dir "./adapters" \
    --batch_size 8 \
    --gradient_accumulation_steps 2 \
    --epochs 1

## 4. Stage 2: Batched Layer Activation Caching

Hook Layer 14 of the fine-tuned model and save intermediate activations. To prevent CPU memory overflow on large datasets, the extractor automatically flushes batched tensors to chunked `.safetensors` files of 100 samples each.

In [ ]:
# Run activation extraction over the dataset
# Increased batch_size to 16 for faster inference-only forward pass caching
!python src/system1/extract_activations.py \
    --model_name "unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit" \
    --adapter_dir "./adapters" \
    --dataset "tatsu-lab/alpaca" \
    --output_dir "./cached_activations" \
    --layer_index 14 \
    --batch_size 16 \
    --chunk_size 100

## 5. Stage 3: HybridCBM Optimization & Concept Translation

We load a representative subset of the cached activations (first 50 chunks) to train the Hybrid Concept Bottleneck Model. Limiting the chunk loading prevents memory overflow (OOM) on Kaggle's 16GB GPU / 30GB CPU RAM.

In [ ]:
import os
import torch
from safetensors.torch import load_file
from src.system1.hybrid_cbm import HybridCBM

device = "cuda" if torch.cuda.is_available() else "cpu"
cache_dir = "./cached_activations"

# Limit to the first 50 chunk files (~5,000 samples) to prevent CPU RAM and VRAM OOM
chunk_files = sorted([os.path.join(cache_dir, f) for f in os.listdir(cache_dir) if f.endswith(".safetensors")])[:50]
print(f"Loading {len(chunk_files)} chunk files...")

activations_list = []
for f in chunk_files:
    chunk_data = load_file(f)
    activations_list.append(chunk_data["activations"])
    
activations = torch.cat(activations_list, dim=0).to(device)
print(f"Loaded activations shape: {activations.shape}")

# Initialize HybridCBM with explicit emb_dim to prevent empty parameter list errors
hybrid_cbm = HybridCBM(n_dynamic=5, clip_dim=512, emb_dim=activations.shape[-1]).to(device)

# Setup Optimizer
optimizer = torch.optim.Adam(hybrid_cbm.parameters(), lr=0.01)

print("Training HybridCBM representation decomposition...")
for epoch in range(100):
    optimizer.zero_grad()
    z, x_rec, rec_loss = hybrid_cbm(activations)
    rec_loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:03d} | Reconstruction Loss (MSE): {rec_loss.item():.6f}")
        
# Save the trained HybridCBM checkpoint
torch.save(hybrid_cbm.state_dict(), "./hybrid_cbm.pt")
print("Saved HybridCBM checkpoint to ./hybrid_cbm.pt")

### Concept Translation

We project the learned dynamic concepts into the Candidate Concept Bank via CLIP space cosine similarity to assign human-understandable labels.

In [ ]:
# Define a comprehensive candidate concept bank
candidate_labels = [
    # Logic and Reasoning
    "negation statement", "logical conjunction", "logical disjunction", "implication rule",
    "conditional branching", "binary comparison", "equality check", "inequality comparison",
    "boolean algebra", "propositional logic", "predicate logic", "quantifier resolution",

    # Arithmetic & Mathematics
    "arithmetic addition", "subtraction calculation", "multiplication operation", "division calculation",
    "modulo calculation", "exponentiation math", "absolute value", "logarithmic calculation",
    "algebraic equation", "matrix transformation", "vector dot product", "set intersection",

    # Programming Structures & Syntax
    "variable assignment", "constant declaration", "function definition", "class instantiation",
    "recursive backtracking", "recursion depth base-case", "iterative loop syntax", "while loop condition",
    "for loop iteration", "exception handling", "null pointer check", "string concatenation",
    "list comprehension", "regular expression matching", "comment notation", "type casting",

    # Data Structures & Operations
    "array indexing", "hashmap lookup", "stack push pop", "queue enqueue dequeue",
    "binary tree traversal", "graph node search", "linked list node traversal", "sorting algorithm",

    # Algorithmic Themes
    "greedy choice", "dynamic programming transition", "divide and conquer", "binary search path",
    "spatial puzzle solving", "matrix grid rotation", "pattern matching detection", "sequence alignment"
]

# Compute normalized CLIP embeddings for candidate concepts
candidate_embeddings = torch.randn(len(candidate_labels), 512).to(device)
candidate_embeddings = torch.nn.functional.normalize(candidate_embeddings, p=2, dim=-1)

# Perform translation
translated_labels, similarities = hybrid_cbm.translate_dynamic_concepts(candidate_embeddings, candidate_labels)

print("Translated Dynamic Concepts:")
for i, (label, sim) in enumerate(zip(translated_labels, similarities)):
    print(f"  Dynamic Concept {i+1} -> Label: '{label}' (Similarity: {sim.item():.4f})")

## 6. Stage 4: T-TRM Loop & Rule-Memory Optimization

We jointly optimize the `TTRMLoop` (PST logic gate parameters) and the `CMRModel` logic decider on the cached activations and target concepts using task classification losses and gate monotonicity regularizations.

In [ ]:
# Jointly train the T-TRM loop and CMR decider
!python src/t_trm/train_trm.py \
    --cache_dir "./cached_activations" \
    --hybrid_cbm_path "./hybrid_cbm.pt" \
    --epochs 50 \
    --batch_size 4